# **DeepFER Face Emotion Recognition - Training on GPU**

This notebook implements the face emotion recognition training pipeline on GPU using Keras 3 with PyTorch backend.

### 1. Diagnostics and Environment Setup

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "torch"

import tensorflow as tf
import keras
from keras import layers, models
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))
print("Keras Version:", keras.__version__)
print("Keras Backend:", keras.config.backend())

### 2. Load and Map Datasets

In [ ]:
train_dir = 'preprocessed_images/train'
val_dir = 'preprocessed_images/validation'

class_names = ['Angry', 'Disgust', 'Fear', 'Happy', 'Neutral', 'Sad', 'Surprise']
IMG_SIZE = (48, 48)
BATCH_SIZE = 64

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels='inferred',
    label_mode='categorical',
    color_mode='grayscale',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_names=[c.lower() for c in class_names],
    shuffle=True
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    labels='inferred',
    label_mode='categorical',
    color_mode='grayscale',
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_names=[c.lower() for c in class_names],
    shuffle=False
)

In [ ]:
data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomTranslation(0.1, 0.1),
])

def preprocess_train(image, label):
    image = tf.image.grayscale_to_rgb(image)
    image = tf.image.resize(image, (128, 128), method='bicubic')
    return image, label

def preprocess_val(image, label):
    image = tf.image.grayscale_to_rgb(image)
    image = tf.image.resize(image, (128, 128), method='bicubic')
    return image, label

train_data = train_ds.map(preprocess_train, num_parallel_calls=tf.data.AUTOTUNE)
val_data = val_ds.map(preprocess_val, num_parallel_calls=tf.data.AUTOTUNE)

train_data = train_data.prefetch(tf.data.AUTOTUNE)
val_data = val_data.prefetch(tf.data.AUTOTUNE)

### 3. Compute Class Weights

In [ ]:
counts = [3993, 436, 4103, 7164, 4982, 4938, 3205]
total = sum(counts)
class_weights = {i: total / (len(counts) * counts[i]) for i in range(len(counts))}
print("Class Weights:", class_weights)

### 4. Build Model (EfficientNetB0 Transfer Learning)

In [ ]:
def create_model():
    base_model = keras.applications.EfficientNetB0(
        input_shape=(128, 128, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False

    inputs = layers.Input(shape=(128, 128, 3))
    x = data_augmentation(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(7, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model, base_model

model, base_model = create_model()
model.summary()

### 5. Training - Phase 1: Classification Head Warmup

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history_head = model.fit(
    train_data,
    validation_data=val_data,
    epochs=5,
    class_weight=class_weights
)

### 6. Training - Phase 2: Fine-Tuning Backbone

In [ ]:
base_model.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=5e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath='emotion_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1
    ),
    keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        verbose=1
    )
]

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=35,
    callbacks=callbacks,
    class_weight=class_weights
)

### 7. Evaluation and Visualizations

In [ ]:
if os.path.exists('emotion_model.keras'):
    model = models.load_model('emotion_model.keras')

y_true = []
y_pred = []

for images, labels in val_data:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

In [ ]:
cm = confusion_matrix(y_true, y_pred)
df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)

plt.figure(figsize=(8, 6))
sns.heatmap(df_cm, annot=True, fmt='d', cmap='Blues')
plt.title('Emotion Recognition Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.tight_layout()
plt.show()